# 03 — Kiểm thử luồng RAG HS và QA

Mỗi tình huống có **đầu vào → phản hồi mong đợi → câu trả lời thực tế → kiểm tra → nguồn**. Ưu tiên CLIProxyAPI với Gemini → GPT-OSS 120B; sau đó fallback sang Qwen/GPT-OSS 20B trên Groq theo `.env`.

Các lượt bổ sung thông tin mô phỏng người dùng trả lời yêu cầu làm rõ: giữ các dữ kiện người dùng trước đó, thêm dữ kiện mới và **truy xuất lại**. Các ca thiếu context và lỗi API được ghi rõ là giả lập. Các câu đủ thông tin lấy từ mẫu Dev; đây là kiểm thử hành vi, không phải đánh giá mù hay benchmark model.

Chọn kernel **Python (AI Agent RAG)** rồi **Run All**. Cần PostgreSQL đang chạy và API key trong `.env`. Câu hỏi và nguồn được gửi tới CLIProxyAPI, hoặc Groq khi fallback, nếu bật `RUN_TESTS`.
`MISSING_PRODUCT_INFO` chỉ hỏi dữ kiện hàng hóa/hồ sơ từ người dùng. `INSUFFICIENT_LEGAL_EVIDENCE` báo thiếu nguồn pháp lý trong retrieval/corpus qua `missing_legal_evidence`; không yêu cầu người dùng cung cấp mã HS, GIR hoặc Chú giải. Đề xuất HS kiểm tra chuỗi graph 4 → 6 → 8 số và citation của wording đã đóng gói. Output cũ đã xóa vì contract thay đổi; các report lưu trước đó vẫn dùng schema lịch sử.


## 1. Khởi tạo và chọn các tình huống cần chạy

Thông số CLIProxyAPI/Groq/DB được đọc từ `.env` qua `config/settings.py`. Các cờ chạy test đọc từ `.env` qua `config/testing.py`: `RAG_TEST_RUN_TESTS`, `RAG_TEST_USE_RETRIEVAL_CACHE`, `RAG_TEST_MIN_TURN_INTERVAL_SECONDS`, `RAG_TEST_CASE_IDS` (ID cách nhau bằng dấu phẩy) và `RAG_TEST_RUN_CUSTOM_CASE`. Câu hỏi tự thử vẫn nhập trực tiếp ở cell bên dưới. Kết quả API đang lưu là lượt chạy trước khi tổ chức lại module; việc chuyển file đã được kiểm tra bằng test và notebook smoke test.

In [ ]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

roots = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    roots.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((p.resolve() for p in roots if (p / "rag/chunking/corpus.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, display
from IPython.utils.io import capture_output

from config.settings import load_environment
from config.testing import GenerationTestConfig
from rag.utils.files import digest, read
from rag.vectordb.postgres import connect
from rag.embedding.bge_m3 import Embedder
from tests.support.generation_scenarios import load_scenarios, run_scenarios, summary_rows
from rag.llm.providers.client import RAGClient
from rag.retrieval.retriever import Retriever
from rag.retrieval.query_planning import QueryPlanner

load_environment(override=True)
test_config = GenerationTestConfig.from_env()
RUN_TESTS = test_config.run_tests
USE_RETRIEVAL_CACHE = test_config.use_retrieval_cache
MIN_TURN_INTERVAL_SECONDS = test_config.min_turn_interval_seconds
CASE_IDS = list(test_config.case_ids)
RUN_CUSTOM_CASE = test_config.run_custom_case
CUSTOM_TASK = "hs"
CUSTOM_QUESTION = "Tôi cần xác định mã HS cho một chiếc nhẫn dùng làm đồ trang sức."
CUSTOM_ADDITIONAL_CONTEXT = ["Nhẫn có nền kim loại cơ bản, chỉ mạ vàng."]
RUN_DIR = ROOT / "artifacts/generation" / datetime.now(timezone.utc).strftime("scenarios_%Y%m%dT%H%M%SZ")

client = RAGClient()
display({
    "Model chính": client.config.model,
    "Chuỗi fallback": list(client.config.fallback_models),
    "Số Groq key dự phòng": len(client.fallback.config.keys),
    "Giới hạn đầu vào (ký tự)": client.config.max_input_chars,
    "Giới hạn context (ký tự)": client.config.max_context_chars,
    "Giới hạn output (token)": client.primary.config.max_output_tokens,
    "Reasoning effort": client.primary.config.reasoning_effort,
    "Timeout (s)": client.primary.config.timeout_seconds,
    "Thời gian chờ tối đa (s)": client.fallback.config.max_wait_seconds,
    "Temperature": client.primary.config.temperature,
    "Kết quả lưu tại": str(RUN_DIR),
})

policy = read(ROOT / "config/unified_retrieval.json")
locked = read(ROOT / "artifacts/experiments/run_unified_accuracy_v5/locked_policy.json")
assert policy == locked, "Retrieval policy differs from the frozen Dev lock"
conn = connect()
conn.execute("SET default_transaction_read_only = on")
active = conn.execute("SELECT snapshot,profile FROM rag.active").fetchone()
assert active == {"snapshot": policy["snapshot"], "profile": policy["profile"]}
with capture_output():
    retriever = Retriever(conn, Embedder(), query_planner=QueryPlanner(client))
print("Retrieval đã sẵn sàng; policy hash:", digest(policy))

In [ ]:
scenarios = load_scenarios(client.config)
if CASE_IDS:
    unknown = set(CASE_IDS) - {c["id"] for c in scenarios}
    if unknown:
        raise ValueError("Unknown scenario IDs: " + ", ".join(sorted(unknown)))
    scenarios = [c for c in scenarios if c["id"] in CASE_IDS]

display(pd.DataFrame([
    {"ID": c["id"], "Tình huống": c["name"], "Task": c["task"], "Số lượt": len(c["turns"]),
     "Mong đợi": " → ".join("/".join(t.get("expected_statuses", [])) or "INPUT_REJECTED" for t in c["turns"])}
    for c in scenarios
]))

## 2. Chạy các ca kiểm thử

- **Đủ thông tin:** HS đề xuất mã kèm nguồn; QA trả lời trực tiếp có citation.
- **Thiếu dữ kiện/ngữ cảnh câu hỏi:** hỏi rõ thông tin còn thiếu, chưa chọn mã HS. Lượt sau cung cấp thêm dữ kiện và truy xuất lại.
- **Thông tin mâu thuẫn:** yêu cầu xác nhận; lượt đính chính phải dùng dữ kiện đã được sửa.
- **Thiếu chứng cứ:** giả lập mất nguồn hoặc ngân sách context quá nhỏ; không gọi LLM khi gói nguồn rỗng. Khôi phục context ở lượt sau.
- **Ngoài phạm vi, prompt injection, câu hỏi rỗng/quá dài:** không bịa câu trả lời hoặc mã HS, từ chối đầu vào không hợp lệ trước retrieval/API.
- **Fallback:** giả lập model chính trả 503; request tới model dự phòng là API thật.

`PASS` nghĩa là đạt các kiểm tra của ca đó. JSON/citation hợp lệ vẫn cần người đọc kiểm tra nguồn có thực sự hỗ trợ kết luận hay không.

`MIN_TURN_INTERVAL_SECONDS` giãn các lượt có gọi LLM; có thể giảm khi tài khoản cho phép. Key vẫn xoay vòng, nhưng các key cùng tổ chức dùng chung [giới hạn Groq](https://console.groq.com/docs/rate-limits). Lỗi API và lỗi kỳ vọng được giữ nguyên trong kết quả.

In [ ]:
report = None
if RUN_TESTS:
    report = run_scenarios(
        scenarios, retriever, client, RUN_DIR,
        use_cache=USE_RETRIEVAL_CACHE,
        min_turn_interval_seconds=MIN_TURN_INTERVAL_SECONDS,
        progress=lambda message: print(message, flush=True),
    )
    print("Đã lưu:", RUN_DIR / "scenarios.json")
else:
    print("Chưa gọi API. Đặt RAG_TEST_RUN_TESTS=true trong .env để thực thi.")

## 3. Tổng hợp kết quả thực tế

In [ ]:
if report:
    display(pd.DataFrame(summary_rows(report)))
    passed = sum(r["passed"] for r in report["rows"])
    print(f"Đạt kỳ vọng: {passed}/{report['turn_count']} lượt; {report['scenario_count']} tình huống.")
    failed = [r for r in report["rows"] if not r["passed"]]
    if failed:
        display(pd.DataFrame([
            {"Tình huống": r["name"], "Lượt": r["turn"], "Kiểm tra chưa đạt": [k for k, v in r["checks"].items() if not v],
             "Lỗi API": r.get("http_status"), "Mã lỗi": r.get("error_code")}
            for r in failed
        ]))

## 4. Đọc từng lượt: câu hỏi, yêu cầu bổ sung, câu trả lời và dẫn nguồn

In [ ]:
def show_turn(row):
    display(Markdown(f"### {row['name']} — lượt {row['turn']}"))
    display(Markdown("**Câu hỏi ban đầu:** " + row["initial_question"][:500]))
    if row["supplements"]:
        display(Markdown("**Thông tin người dùng bổ sung:**\n\n" + "\n\n".join(row["supplements"])))
    display({
        "Chế độ": row["mode"],
        "Mong đợi": row["expected"].get("expected_statuses") or row["expected"].get("expected_error"),
        "Kết quả kiểm tra": "PASS" if row["passed"] else "FAIL",
        "Các kiểm tra": row["checks"],
    })
    if row["run_status"] == "INPUT_REJECTED":
        display({"Đã từ chối đầu vào": row["input_error"], "Số ký tự": len(row["initial_question"])})
        return
    if row["run_status"] == "PROVIDER_ERROR":
        display({"HTTP": row["http_status"], "Mã lỗi": row.get("error_code"), "Retry-After": row["retry_after"]})
        return
    if not row["validation"]["valid"]:
        display({"Phản hồi bị chặn bởi validation": row["validation"]["errors"]})
        return
    answer = row["answer"]
    display(Markdown("**Câu trả lời:**\n\n" + answer.get("answer", "")))
    display({"Trạng thái": answer.get("status"), "Mã HS": answer.get("suggested_hs_code"),
             "Model thực tế": row["generation"].get("model"), "Đã dùng fallback": row["generation"].get("fallback_used", False)})
    failures = row["generation"].get("failures", [])
    if failures:
        display({"Lỗi model trước khi có phản hồi": failures})
    if answer.get("status") == "MISSING_PRODUCT_INFO" and answer.get("missing_information"):
        display(Markdown("**Thông tin hàng hóa/hồ sơ người dùng cần bổ sung:**\n\n" + "\n".join("- " + item for item in answer["missing_information"])))
    if answer.get("missing_legal_evidence"):
        display(Markdown("**Khoảng trống chứng cứ của hệ thống (retrieval/corpus):**\n\n" + "\n".join("- " + item for item in answer["missing_legal_evidence"])))
    gold = row["expected"].get("gold_case", {})
    if gold.get("expected_hs_8"):
        display({"Mã gold dùng kiểm tra sau generation": gold["expected_hs_8"]})
    sources = {s["label"]: s for s in row["context"]["sources"]}
    if answer.get("citations"):
        display(pd.DataFrame([
            {"Nhãn": c["label"], "Trích nguyên văn": c["quote"], "Nguồn": sources.get(c["label"], {}).get("id"),
             "Tài liệu/trang": sources.get(c["label"], {}).get("locator")}
            for c in answer["citations"]
        ]))
    else:
        print("Không có citation trong lượt này.")
    print("Số nguồn gửi LLM:", len(row["context"]["sources"]), "— Số nguồn bị bỏ:", len(row["context"]["omitted_source_ids"]))

if report:
    for row in report["rows"]:
        show_turn(row)

## 5. Tự thử câu hỏi và thông tin bổ sung

Sửa `CUSTOM_TASK`, `CUSTOM_QUESTION`, `CUSTOM_ADDITIONAL_CONTEXT` ở cell đầu, rồi đặt `RAG_TEST_RUN_CUSTOM_CASE=true` trong `.env`. Mỗi phần tử thông tin bổ sung là một lượt người dùng mới, luôn truy xuất lại. Ca tự thử chỉ kiểm tra schema/citation; bạn cần tự đánh giá nội dung.

In [ ]:
if RUN_CUSTOM_CASE:
    custom_case = {
        "id": "custom", "name": "Câu hỏi tự thử", "task": CUSTOM_TASK,
        "question": CUSTOM_QUESTION,
        "turns": [{}, *[{"supplement": text} for text in CUSTOM_ADDITIONAL_CONTEXT]],
    }
    custom_report = run_scenarios([custom_case], retriever, client, RUN_DIR / "custom", use_cache=USE_RETRIEVAL_CACHE, min_turn_interval_seconds=MIN_TURN_INTERVAL_SECONDS)
    for row in custom_report["rows"]:
        show_turn(row)

## 6. Rà soát và kết thúc

Report giữ câu hỏi đã ghép, nguồn, nguồn bị bỏ, prompt/schema hashes, model thực tế và từng kiểm tra. Gold chỉ dùng sau generation. Các ca thiếu context/lỗi model chính là fault injection có ghi rõ; không phải sự cố thật của PostgreSQL, CLIProxyAPI hoặc Groq. Độ đúng và độ đầy đủ của câu trả lời QA, cũng như việc nguồn hỗ trợ mọi kết luận, cần rà soát thủ công.

In [ ]:
if report:
    review = pd.DataFrame([
        {"scenario_id": r["scenario_id"], "turn": r["turn"], "automated_pass": r["passed"],
         "answer_correct": None, "citations_support_claims": None,
         "clarification_relevant": None, "unsupported_assumptions": None, "notes": ""}
        for r in report["rows"] if r.get("answer")
    ])
    review.to_csv(RUN_DIR / "manual_review.csv", index=False)
    print("Bảng rà soát:", RUN_DIR / "manual_review.csv")
conn.close()